# 01 English Dataset EDA

Quick sanity-check notebook for the Kaggle dataset `rayyankauchali0/resume-dataset`.

What it does:
- downloads and loads the dataset via `kagglehub`
- inspects schema and sample rows
- checks source composition (`real` / `template` / `llm` / `faker`)
- checks label and location distributions
- builds a raw `resume_text` field if needed
- estimates whether the dataset is usable as a curated English pilot

Default flow:
1. Run the notebook as is.
2. It downloads the latest Kaggle dataset via `kagglehub.dataset_download("rayyankauchali0/resume-dataset")`.
3. The notebook then finds the downloaded data file and loads it for EDA.


In [ ]:
import json
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)


In [ ]:
KAGGLE_HANDLE = "rayyankauchali0/resume-dataset"

# Download latest version
path = kagglehub.dataset_download(KAGGLE_HANDLE)
downloaded_path = Path(path)

print("Path to dataset files:", downloaded_path)

SEARCH_DIRS = [
    downloaded_path if downloaded_path.is_dir() else downloaded_path.parent,
]

KNOWN_FILENAMES = {
    "tech_resumes_dataset.jsonl",
    "Resume_Dataset.zip",
    "resume-dataset.zip",
}

def find_candidate_files(search_dirs):
    candidates = []
    for base in search_dirs:
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if not path.is_file():
                continue
            if path.name in KNOWN_FILENAMES or path.suffix.lower() in {".jsonl", ".csv", ".zip"}:
                candidates.append(path)
    return sorted(set(candidates))

candidates = find_candidate_files(SEARCH_DIRS)

print("Search dirs:")
for d in SEARCH_DIRS:
    print(" -", d)
print("\nCandidates:")
for c in candidates[:30]:
    print(" -", c)

if not candidates:
    raise FileNotFoundError("No dataset file found in the downloaded Kaggle dataset directory")


In [ ]:
def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

def load_from_zip(path):
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        jsonl_names = [n for n in names if n.lower().endswith('.jsonl')]
        csv_names = [n for n in names if n.lower().endswith('.csv')]

        if jsonl_names:
            target = jsonl_names[0]
            with zf.open(target) as f:
                rows = [json.loads(line.decode('utf-8')) for line in f if line.strip()]
            return pd.DataFrame(rows), target

        if csv_names:
            target = csv_names[0]
            with zf.open(target) as f:
                return pd.read_csv(f), target

        raise FileNotFoundError("No .jsonl or .csv found inside archive")

dataset_path = None
dataset_inner_name = None
df = None

for candidate in candidates:
    try:
        if candidate.suffix.lower() == ".jsonl":
            df = load_jsonl(candidate)
            dataset_path = candidate
            break
        if candidate.suffix.lower() == ".csv":
            df = pd.read_csv(candidate)
            dataset_path = candidate
            break
        if candidate.suffix.lower() == ".zip":
            df, dataset_inner_name = load_from_zip(candidate)
            dataset_path = candidate
            break
    except Exception as exc:
        print(f"Skipping {candidate}: {exc}")

if df is None:
    raise RuntimeError("Found candidate files, but failed to load any dataset")

print("Loaded from:", dataset_path)
if dataset_inner_name is not None:
    print("Inner file:", dataset_inner_name)
print("Shape:", df.shape)


In [ ]:
print("Columns:")
for col in df.columns:
    print(" -", col)

display(df.head(3))
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_share").head(20))


In [ ]:
TEXT_CANDIDATES = ["Resume_Text", "resume_text", "Summary", "Experience", "Education", "Skills"]
CATEGORY_CANDIDATES = ["Category", "category", "Job_Role", "job_role"]
LOCATION_CANDIDATES = ["Location", "location", "City", "city"]
SOURCE_CANDIDATES = ["Source", "source"]

def first_existing(candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

category_col = first_existing(CATEGORY_CANDIDATES)
location_col = first_existing(LOCATION_CANDIDATES)
source_col = first_existing(SOURCE_CANDIDATES)
resume_text_col = first_existing(TEXT_CANDIDATES)

if resume_text_col is None:
    text_parts = [c for c in ["Summary", "Skills", "Experience", "Education"] if c in df.columns]
    if text_parts:
        df["resume_text"] = df[text_parts].fillna("").astype(str).agg("\n\n".join, axis=1)
        resume_text_col = "resume_text"

print("category_col:", category_col)
print("location_col:", location_col)
print("source_col:", source_col)
print("resume_text_col:", resume_text_col)

if resume_text_col is None:
    raise KeyError("Could not find or construct a resume text column")


In [ ]:
df["resume_text_work"] = df[resume_text_col].fillna("").astype(str)
df["text_len_chars"] = df["resume_text_work"].str.len()
df["text_len_words"] = df["resume_text_work"].str.split().str.len()

summary = {
    "n_rows": len(df),
    "n_cols": len(df.columns),
    "empty_text_share": float((df["resume_text_work"].str.strip() == "").mean()),
    "median_chars": float(df["text_len_chars"].median()),
    "median_words": float(df["text_len_words"].median()),
    "p10_words": float(df["text_len_words"].quantile(0.10)),
    "p90_words": float(df["text_len_words"].quantile(0.90)),
}
summary


In [ ]:
if category_col is not None:
    cat_counts = df[category_col].value_counts(dropna=False)
    display(cat_counts.head(30).to_frame("count"))
    print("Unique categories:", df[category_col].nunique(dropna=True))

if source_col is not None:
    source_counts = df[source_col].value_counts(dropna=False)
    display(source_counts.to_frame("count"))
    if source_counts.sum() > 0:
        display((source_counts / source_counts.sum()).round(4).to_frame("share"))

if location_col is not None:
    loc_counts = df[location_col].astype(str).value_counts(dropna=False)
    display(loc_counts.head(30).to_frame("count"))
    print("Unique locations:", df[location_col].nunique(dropna=True))


In [ ]:
sample_cols = [c for c in [category_col, location_col, source_col, "Summary", "Skills", "Experience", "Education", "resume_text_work"] if c is not None and c in df.columns]
sample_df = df[sample_cols].sample(min(8, len(df)), random_state=42)
display(sample_df)


In [ ]:
TARGET_9_CLASSES = [
    "backend_general_dev",
    "web_frontend",
    "sysadmin_devops_network",
    "project_product",
    "sales_account",
    "tech_support_helpdesk",
    "it_governance_leadership",
    "technical_specialized",
    "generic_it_ops",
]

print("Target classes in your current project:")
for c in TARGET_9_CLASSES:
    print(" -", c)

if category_col is not None:
    unique_cats = sorted(df[category_col].dropna().astype(str).unique())
    print("\nFirst 60 raw English categories:")
    for c in unique_cats[:60]:
        print(" -", c)


In [ ]:
checks = []

checks.append({
    "check": "Has explicit category column",
    "status": category_col is not None,
})
checks.append({
    "check": "Has explicit location column",
    "status": location_col is not None,
})
checks.append({
    "check": "Has explicit source column",
    "status": source_col is not None,
})
checks.append({
    "check": "Median text length >= 80 words",
    "status": summary["median_words"] >= 80,
})

if source_col is not None:
    synthetic_like = df[source_col].astype(str).str.lower().isin(["template", "llm", "faker"]).mean()
    checks.append({
        "check": "Synthetic-like share < 0.70",
        "status": synthetic_like < 0.70,
        "value": round(float(synthetic_like), 4),
    })

checks_df = pd.DataFrame(checks)
display(checks_df)


In [ ]:
print("Quick verdict:")
print("- If text fields look real enough and categories seem mappable, this dataset is usable as a curated EN pilot.")
print("- If source distribution is dominated by template/llm/faker, use it only as a weak exploratory benchmark.")
print("- Next step after this notebook: manually inspect 30-50 resumes and draft a category mapping into your 9-class ontology.")
